# Deep AutoEncoder for Log Anomaly Detection

This notebook combines structured log features with semantic LogBERT embeddings.

A Deep AutoEncoder is trained to reconstruct normal log behaviour. Logs that cannot be reconstructed well receive higher reconstruction errors and are marked as anomalies.

Pipeline

Processed Logs
+
LogBERT Embeddings
↓

Feature Engineering

↓

Normalization

↓

Deep AutoEncoder

↓

Reconstruction Error

↓

Anomaly Detection

In [2]:
# ================================
# 📚 Imports
# ================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

# Scikit-Learn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Progress Bar
from tqdm import tqdm

# Metrics
from sklearn.metrics import mean_squared_error

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

In [3]:
# ================================
# Device Configuration
# ================================

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using Device: {device}")

Using Device: mps


In [4]:
# ================================
# Random Seed
# ================================

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

print("Random seed set successfully.")

Random seed set successfully.


## Load Processed Logs + LogBERT Embeddings

## Load Structured Logs and LogBERT Embeddings

In this section, we load:

- Structured log features generated during preprocessing.
- Semantic embeddings generated using LogBERT.

These two sources of information will later be merged to create a rich feature representation for anomaly detection.

In [6]:
# ================================
# Load Data
# ================================

logs = pd.read_csv("../data/processed/processed_logs.csv")

embeddings = np.load("../data/embeddings/logbert_embeddings.npy")

print("Structured Logs Shape :", logs.shape)
print("Embeddings Shape      :", embeddings.shape)

Structured Logs Shape : (4945, 23)
Embeddings Shape      : (4945, 768)


In [7]:
# Preview Structured Logs

logs.head()


,Raw_Log,Component,Line_Number,Timestamp,Severity,Username,AccountID,Originating_Address,Destination_Address,TPS,...,PositiveResponse,QueueInserted,RouterSent,ContainsError,Message_Length,Word_Count,Hour,Minute,Second,Millisecond
0,[ESME.cpp|01321|02:46:17:014|~CR~][CBS_Product...,ESME.cpp,1321,1900-01-01 02:46:17.014,~CR~,0,0.0,0,0,0.0,...,1,0,0,0,99,3,2,46,17,14
1,[ESME.cpp|00448|02:46:17:059|~CR~][SimulQ2.824...,ESME.cpp,448,1900-01-01 02:46:17.059,~CR~,0,0.0,0,0,0.0,...,0,1,0,0,116,6,2,46,17,59
2,[Globals.cpp|00107|02:46:17:059|~CR~]Checkign ...,Globals.cpp,107,1900-01-01 02:46:17.059,~CR~,SimulQ2,0.0,Yas,0,0.0,...,0,0,0,0,73,3,2,46,17,59
3,[Globals.cpp|00124|02:46:17:060|~CR~]Checkign ...,Globals.cpp,124,1900-01-01 02:46:17.060,~CR~,0,0.0,0,255653530543,0.0,...,0,0,0,0,77,3,2,46,17,60
4,[ESME.cpp|04785|02:46:17:060|~CR~]A2P Authenti...,ESME.cpp,4785,1900-01-01 02:46:17.060,~CR~,0,0.0,0,0,0.0,...,0,0,0,0,60,3,2,46,17,60


In [8]:
# Preview Embedding Matrix

print(embeddings[:3])


[[-0.59747344 -0.21827132  0.01647669 ... -0.43211138  0.29980865
   0.5869379 ]
 [-0.52149326 -0.2856987  -0.2624236  ... -0.18655363 -0.0100332
   0.88385147]
 [-0.43218356 -0.06748354  0.01606561 ... -0.4886317   0.3148201
   0.7241904 ]]
